# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



## Setup: Create Agent

In [1]:
import os
os.environ["LOG_LEVEL"] = "WARNING"

In [2]:
from gai.lib.constants import DEFAULT_GUID
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.messages import FileMonologue
from gai.messages import FileDialogue, MessagePydantic

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

def print_chunk(chunk):
    """
    Helper function to print the chunk of data received from the agent.
    """
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
        else:
            if isinstance(chunk, list):
                for item in chunk:
                    if item.get("name"):
                        print(f'Tool: "{item["name"]}"')
                    if item.get("input"):
                        inputs = item.get("input")
                        if isinstance(inputs, dict):
                            for key, value in inputs.items():
                                if isinstance(value, str):
                                    if len(value) > 100:
                                        print(
                                            f"\tInput: {key} = {value[:100]}... (truncated)"
                                        )
                                    else:
                                        print(f"\tInput: {key} = {value}")

---

## Scenario 1: Context inferred from dialogue context

In this scenario, we create a hypothetical conversation where the user talks about weather in Singapore.  
Then the user asked about the time.  
The agent should infer that user is asking about the time in Singapore and immediately proceed to run the tool to get the time in Singapore.

### a) start




In [3]:
# Reset the monologue to start fresh

monologue.reset()

# Create an artificial dialogue history for testing
messages = [
    MessagePydantic(
        **{
            "id": "b1e5f98c-f6eb-47de-a6e2-387510d970f9",
            "header": {
                "sender": "User",
                "recipient": "Sara",
                "timestamp": 1751308157.270983,
                "order": 0,
            },
            "body": {
                "type": "chat.send",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 0,
                "role": "user",
                "content": "It is a very nice weather in Singapore right now.",
            },
        }
    ),
    MessagePydantic(
        **{
            "id": "abbc7961-45dc-4973-aaf4-a6224ed35d37",
            "header": {
                "sender": "Sara",
                "recipient": "User",
                "timestamp": 1751308167.3488164,
                "order": 1,
            },
            "body": {
                "type": "chat.reply",
                "dialogue_id": "00000000-0000-0000-0000-000000000000",
                "round_no": 0,
                "step_no": 1,
                "chunk_no": 10,
                "chunk": "<eom>",
                "role": "assistant",
                "content": "Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?",
            },
        }
    ),
]
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages, file_path=file_path)

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue
)

goal = "When is the next public holiday?"
user_message=f"""
## Goal

{goal}

## Instructions

- Find the best way to achieve the goal.
"""
recap = dialogue.extract_recap()
resp=agent.start(user_message=user_message, recap=recap)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue
dialogue.add_user_message(recipient="Sara", content=goal)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)



Hello! I'm ToolUseAgent, and I'll help you find information about the next public holiday.

To find the next public holiday, I'll need to search for current public holiday information. Let me do that for you.
Tool: "search"
	Input: search_query = next public holiday 2024 2025 Singapore
Let me get the current date to determine which is the next upcoming public holiday:
Tool: "current_time"
	Input: format = YYYY-MM-DD


MessagePydantic(id='1714021b-5b59-41d1-9b80-bda1ea888e96', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753271921.3443494, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.39', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content='Let me get the current date to determine which is the next upcoming public holiday:'))

### b) resume

In [4]:
resp = agent.resume()
# Stream the response
can_resume=False
async for chunk in resp:
    can_resume=True
    print_chunk(chunk)
if not can_resume:
    raise Exception("Agent should be able to resume without user input")


Based on my search results and the current date (July 23, 2025), I can see that the **next public holiday in Singapore is National Day on Saturday, August 9, 2025**.

Here's a summary of the upcoming public holidays in Singapore for the remainder of 2025:

1. **National Day** - Saturday, August 9, 2025 (next upcoming holiday)
2. **Deepavali** - Monday, October 20, 2025  
3. **Christmas Day** - Thursday, December 25, 2025

Since National Day falls on a Saturday this year, employers typically provide an additional day off in lieu for their employees, which would likely be on the following Monday (August 11, 2025).

So the next public holiday is **National Day on August 9, 2025** - that's about 2.5 weeks away!


### c) Show monologue

See the monologue for details at `tmp/monologue.json`

In [5]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "f1d24c4c-c67d-4e4a-afec-1f9052332954",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1753271931.1201704,
        "order": 4
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 7,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_01D4StSTDditiZu1cdJcsHxv",
                "content": "Current UTC time is 2025-07-23, and the time in UTC is 2025-07-23."
            }
        ]
    }
}
{
    "id": "99453492-7bee-49f5-b9f9-d8c50a232dd6",
    "header": {
        "sender": "ToolUseAgent",
        "recipient": "User",
        "timestamp": 1753271940.2496197,
        "order": 5
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 7,
        "conte

---

## Scenario 2: Agent interrupt user for input

In this scenario, we just ask the agent directly about the time without giving any context. The agent should ask the user for input to continue the conversation.

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

### a) start


In [3]:
monologue.reset()

file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(file_path=file_path)

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
)

goal = "When is the next public holiday?"
user_message=f"""
## Goal

{goal}

## Instructions

- Feel free to ask for more information if needed.
"""

resp=agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue
dialogue.add_user_message(recipient="Sara", content=goal)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Hello! I'm ToolUseAgent. To help you find the next public holiday, I'll need a bit more information:

1. Which country or region are you asking about? Public holidays vary significantly by location.

2. What's your current location or the location you're interested in?

Once you provide this information, I can search for the upcoming public holidays in that specific area.
Tool: "user_input"
I notice you've repeated the same question. To find the next public holiday for you, I still need to know which country or region you're asking about, since public holidays are different in each location. 

Could you please tell me:
- Which country are you in?
- Or which country's public holidays are you interested in?

For example, are you asking about holidays in the United States, United Kingdom, Canada, Australia, or another country?


MessagePydantic(id='011007a4-5a2c-480a-83f4-d3ea60b1da4a', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753272029.8296318, order=1), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=0, step_no=1, message_id='00000000-0000-0000-0000-000000000000.31', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content="I notice you've repeated the same question. To find the next public holiday for you, I still need to know which country or region you're asking about, since public holidays are different in each location. \n\nCould you please tell me:\n- Which country are you in?\n- Or which country's public holidays are you interested in?\n\nFor example, are you asking about holidays in the United States, United Kingdom, Canada, Australia, or another country?"))

### d) Unable to resume without user input

In [4]:
resp = agent.resume()
# Stream the response
can_resume=False
async for chunk in resp:
    can_resume=True
    print_chunk(chunk)
if not can_resume:
    print("Cannot resume because user has not provided input.")
assert not can_resume, "Agent should not be able to resume while waiting for input."


Cannot resume because user has not provided input.


### e) Can resume normally after user input

In [5]:
# Resume with the user input and for the agent to create a tool use
resp = agent.resume("Use SGT")
async for chunk in resp:
    print_chunk(chunk)

# Resume again (normal) to return the tool use result
resp = agent.resume()
async for chunk in resp:
    print_chunk(chunk)

Thank you for clarifying! You're asking about public holidays in Singapore (SGT - Singapore Time). Let me search for the upcoming public holidays in Singapore.
Tool: "current_time"
	Input: format = YYYY-MM-DD
	Input: timezone = Asia/Singapore
Thinking...
Tool: "search"
	Input: search_query = Singapore public holidays 2025 upcoming next holiday dates
Based on the search results from Singapore's Ministry of Manpower, I can see the upcoming public holidays for 2025. Since today is July 23, 2025, the **next public holiday in Singapore is National Day on Saturday, August 9, 2025**.

Here's the complete list of remaining public holidays for 2025 in Singapore:

1. **National Day** - Saturday, August 9, 2025
2. **Deepavali** - Monday, October 20, 2025  
3. **Christmas Day** - Thursday, December 25, 2025

So your next public holiday is just about 2.5 weeks away - Singapore's National Day on August 9th!


---

## Scenario 3: User interrupt agent with adhoc input

In this scenario, the user tries to distract the agent by interrupting it with an adhoc input.

### a) start


In [6]:
monologue.reset()

file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(file_path=file_path)

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
)

goal = "When is the next public holiday?"
user_message=f"""
## Goal

{goal}


"""

resp=agent.start(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue
dialogue.add_user_message(recipient="Sara", content=goal)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

I'll help you find information about the next public holiday. Let me search for current public holiday information.
Tool: "search"
	Input: search_query = next public holiday 2024 2025 upcoming
I notice the search results are showing Singapore public holidays. Let me get the current date first to determine which holiday is actually next, and then search for more general public holiday information.
Tool: "current_time"
	Input: format = YYYY-MM-DD


MessagePydantic(id='0932bf45-fe49-45cf-a668-32e0349d3263', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753272166.5349991, order=3), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=1, step_no=1, message_id='00000000-0000-0000-0000-000000000000.235', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content='I notice the search results are showing Singapore public holidays. Let me get the current date first to determine which holiday is actually next, and then search for more general public holiday information.'))

### b) User interrupt agent

Should be able to interrupt the agent and continue with original task.

In [7]:
user_message = "Tell me a one paragraph joke."
resp = agent.interrupt(user_message=user_message)
# Stream the response
async for chunk in resp:
    print_chunk(chunk)

# Update dialogue when user interrupt agent
dialogue.add_user_message(recipient="Sara", content=user_message)
dialogue.add_assistant_message(
    sender="Sara", chunk="<eom>", content=agent.final_output()
)

Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, sees nothing, and turns back to find the librarian has vanished. He stands there confused until he hears a faint voice from behind the counter saying, "Sorry, I was just getting your books - turns out we keep all the paranoia books in the back section because we're worried people might be watching them too closely. Also, I should mention that everyone who checks out these books gets followed home by our security team, but don't worry, it's just to make sure you're reading them properly and not sharing the secrets with anyone suspicious."

Now, getting back to your original question about the next public holiday - let me get the current date and provide you with accurate information about upcoming public holidays.
Thinking...


MessagePydantic(id='a5bea499-05d1-4a49-b6ed-02671553f906', header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1753272208.706365, order=5), body=ChatReplyBodyPydantic(type='chat.reply', dialogue_id='00000000-0000-0000-0000-000000000000', round_no=2, step_no=1, message_id='00000000-0000-0000-0000-000000000000.342', chunk_no=0, chunk='<eom>', content_type='text', role='assistant', content='Here\'s a one paragraph joke for you:\n\nA man walks into a library and asks for books on paranoia. The librarian whispers, "They\'re right behind you!" The man spins around frantically, sees nothing, and turns back to find the librarian has vanished. He stands there confused until he hears a faint voice from behind the counter saying, "Sorry, I was just getting your books - turns out we keep all the paranoia books in the back section because we\'re worried people might be watching them too closely. Also, I should mention that everyone who checks out these books gets followed home b

### d) Show dialogue

In [8]:
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: Sara, When is the next public holiday?
Sara: I notice you've repeated the same question. To find the next public holiday for you, I still need to know which country or region you're asking about, since public holidays are different in each location. 

Could you please tell me:
- Which country are you in?
- Or which country's public holidays are you interested in?

For example, are you asking about holidays in the United States, United Kingdom, Canada, Australia, or another country?
User: Sara, When is the next public holiday?
Sara: I notice the search results are showing Singapore public holidays. Let me get the current date first to determine which holiday is actually next, and then search for more general public holiday information.
User: Sara, Tell me a one paragraph joke.
Sara: Here's a one paragraph joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, sees nothing, and 